In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# QSS

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}")

DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


MASE_MODEL_DIRS = {
    "ARIMA": BASE_DIR / "arima_standardized",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "LSTM": BASE_DIR / "lstm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


# Quantile-file suffix used by each model.
MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
    "ARIMA": "_arima",
    "LSTM": "_lstm",
}


# ============================================================
# PINBALL LOSS
# ============================================================

def pinball_loss(y, q, tau=0.1):
    """
    Pinball loss rho_tau(y - q).

    y, q must have identical shapes.
    """

    y = np.asarray(y, dtype=float)
    q = np.asarray(q, dtype=float)

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


# ============================================================
# DATA HELPERS
# ============================================================

def load_test_xy(test_path):
    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def reference_q10(X):
    """
    Rolling empirical 10% quantile using ONLY
    the historical context X.
    """

    return np.quantile(
        X,
        TAU,
        axis=1,
    )


# ============================================================
# QSS
# ============================================================

def calculate_qss(
    X,
    Y,
    model_quantiles,
    quantile_levels,
):
    """
    Returns:
        overall model QL
        overall reference QL
        overall QSS
        QSS by horizon
        model QL by horizon
        reference QL by horizon
    """

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    Q = np.asarray(model_quantiles, dtype=float)

    quantile_levels = np.asarray(
        quantile_levels,
        dtype=float,
    )

    # --------------------------------------------------------
    # Locate tau = 0.1
    # --------------------------------------------------------

    q10_idx = np.argmin(
        np.abs(quantile_levels - TAU)
    )

    if not np.isclose(
        quantile_levels[q10_idx],
        TAU,
    ):
        raise ValueError(
            f"No tau={TAU} quantile found."
        )

    model_q10 = Q[:, :, q10_idx]

    if model_q10.shape != Y.shape:
        raise ValueError(
            f"Model q10 shape {model_q10.shape} "
            f"does not match Y shape {Y.shape}"
        )

    # --------------------------------------------------------
    # Model pinball losses
    # --------------------------------------------------------

    model_loss = pinball_loss(
        Y,
        model_q10,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Reference q10:
    #
    # one empirical 10th percentile per forecast origin
    # calculated from X_0,...,X_249 only
    # --------------------------------------------------------

    q_ref = reference_q10(X)

    # Same reference forecast for every future horizon
    q_ref = q_ref[:, None]

    reference_loss = pinball_loss(
        Y,
        q_ref,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Horizon-specific quantities
    # --------------------------------------------------------

    model_ql_by_horizon = np.mean(
        model_loss,
        axis=0,
    )

    ref_ql_by_horizon = np.mean(
        reference_loss,
        axis=0,
    )

    qss_by_horizon = (
        1.0
        - model_ql_by_horizon
        / ref_ql_by_horizon
    )

    # --------------------------------------------------------
    # Overall quantities
    #
    # IMPORTANT:
    # calculate ratio of aggregate losses,
    # NOT mean(qss_by_horizon)
    # --------------------------------------------------------

    model_ql = float(
        np.mean(model_loss)
    )

    ref_ql = float(
        np.mean(reference_loss)
    )

    qss = float(
        1.0
        - model_ql / ref_ql
    )

    return {
        "model_ql": model_ql,
        "reference_ql": ref_ql,
        "qss": qss,
        "model_ql_by_horizon": model_ql_by_horizon,
        "reference_ql_by_horizon": ref_ql_by_horizon,
        "qss_by_horizon": qss_by_horizon,
    }


# ============================================================
# FIND FILES
# ============================================================

def find_test_file(asset_class, dataset_stem):
    """
    Find the original held-out test CSV corresponding
    to the dataset.
    """

    asset_dir = DATA_DIR / asset_class

    matches = list(
        asset_dir.glob(
            f"{dataset_stem}.csv"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely identify test file for "
        f"{dataset_stem} in {asset_class}. "
        f"Found {len(matches)} matches."
    )


def dataset_stem_from_quantile_file(
    quantile_path,
    model_suffix,
):
    """
    Example:

    test_Bitcoin Historical Data_no_preprocess_...
        _lightgbm_quantiles.npz

    becomes

    test_Bitcoin Historical Data_no_preprocess_...
    """

    suffix = (
        f"{model_suffix}_quantiles.npz"
    )

    name = quantile_path.name

    if not name.endswith(suffix):
        raise ValueError(
            f"Unexpected quantile filename: {name}"
        )

    return name[:-len(suffix)]

def clean_garch_dataset_name(dataset_name):
    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace("_no_preprocess_lookback-250_horizon-20_slide-1", "")
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):
    """
    Locate GARCH(1,1) quantile file using the
    original GARCH filename convention.
    """

    asset_dir = model_root / asset_class

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    folder_name = asset_class

    prefix = (
        f"{folder_name.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    # Fallback in case there are naming variations
    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No GARCH quantile file found for "
            f"{dataset_name} in {asset_class}"
        )

    raise RuntimeError(
        f"Multiple GARCH quantile files found for "
        f"{dataset_name}: {matches}"
    )


# ============================================================
# CALCULATE QSS FOR ALL MODELS
# ============================================================

qss_summary_rows = []
qss_horizon_rows = []


for model_name, model_root in MODEL_DIRS.items():

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    model_suffix = MODEL_SUFFIXES[model_name]

    for asset_class, asset_name in ASSET_ORDER.items():

        model_asset_dir = model_root / asset_class

        test_asset_dir = DATA_DIR / asset_class

        test_files = sorted(
            test_asset_dir.glob("test_*.csv")
        )

        for test_path in test_files:

            try:
    
                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )
    
                # --------------------------------------------
                # Test X/Y
                # --------------------------------------------
    
                X, Y = load_test_xy(
                    test_path
                )
    
                # --------------------------------------------
                # Model quantiles
                # --------------------------------------------
    
                if model_name == "GARCH(1,1)":
                    garch_path = find_garch_quantile_file(
                        MODEL_DIRS["GARCH(1,1)"],
                        asset_class,
                        dataset_name,
                    )
    
                    quantile_path = garch_path
                else:
                    matches = list(
                        model_asset_dir.glob(
                            f"test_{dataset_name}{model_suffix}_quantiles.npz"
                        )
                    )
                
                    if len(matches) == 1:
                        quantile_path = matches[0]
                    else:
                        raise FileNotFoundError(
                            f"Could not uniquely identify test quantiles for "
                            f"{dataset_name} in {asset_class}. "
                            f"Found {len(matches)} matches."
                        ) 
                    
                with np.load(
                    quantile_path
                ) as q_file:
    
                    quantiles = np.asarray(
                        q_file["quantiles"],
                        dtype=float,
                    )
        
                    quantile_levels = np.asarray(
                        q_file["quantile_levels"],
                        dtype=float,
                    )
    
                # --------------------------------------------
                # QSS
                # --------------------------------------------
    
                result = calculate_qss(
                    X=X,
                    Y=Y,
                    model_quantiles=quantiles,
                    quantile_levels=quantile_levels,
                )
    
                qss_summary_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "model_ql_0.1": (
                        result["model_ql"]
                    ),
                    "reference_ql_0.1": (
                        result["reference_ql"]
                    ),
                    "qss_0.1": (
                        result["qss"]
                    ),
                })
    
                for h in range(HORIZON):
    
                    qss_horizon_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "horizon": h + 1,
                        "model_ql_0.1": (
                            result[
                                "model_ql_by_horizon"
                            ][h]
                        ),
                        "reference_ql_0.1": (
                            result[
                                "reference_ql_by_horizon"
                            ][h]
                        ),
                        "qss_0.1": (
                            result[
                                "qss_by_horizon"
                            ][h]
                        ),
                    })
    
                print(
                    f"  {dataset_name}: "
                    f"QSS={result['qss']:.6f}"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# SAVE QSS
# ============================================================

qss_summary_df = pd.DataFrame(
    qss_summary_rows
)

qss_horizon_df = pd.DataFrame(
    qss_horizon_rows
)

qss_summary_df.to_csv(
    BASE_DIR / "qss_summary_all_models.csv",
    index=False,
)

qss_horizon_df.to_csv(
    BASE_DIR / "qss_by_horizon_all_models.csv",
    index=False,
)


print()
print("Saved QSS outputs.")

# RMSE

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

Y_COLS = [
    f"Y_{h}"
    for h in range(HORIZON)
]

# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}")

DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")



MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


POINT_MODEL_DIRS = {
    "ARIMA": BASE_DIR / "arima_standardized",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "LSTM": BASE_DIR / "lstm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


# point_file suffix used by each model.
MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial_zero-shot",
    "Lag-Llama": "_lag-llama_zero-shot",
    "Chronos-T5": "_chronos-t5-small_zero-shot",
    "Toto": "_toto_zero-shot",
    "TimesFM": "_timesfm-2.5-200m_zero-shot",
    "Moirai": "_moirai-2.0-r-small_zero-shot",
    "Moirai-MoE": "_moirai-moe-1.0-r-base_zero-shot",
    "ARIMA": "_arima",
    "LSTM": "_lstm",
}


rmse_summary_rows = []
rmse_horizon_rows = []


for model_name, model_root in POINT_MODEL_DIRS.items():

    print()
    print(f"Calculating RMSE: {model_name}")
    model_suffix = MODEL_SUFFIXES[model_name]

    for asset_class, asset_name in ASSET_ORDER.items():

        asset_dir = model_root / asset_class

        if not asset_dir.exists():
            print(f"  Missing: {asset_dir}")
            continue

        point_files = sorted(
            asset_dir.glob(f"test_*{model_suffix}.csv")
        )

        print(
            f"  {asset_class}: "
            f"{len(point_files)} point files"
        )

        for point_path in point_files:

            try:

                # ----------------------------------------
                # Extract / match dataset name here
                # ----------------------------------------
                dataset_name = point_path.stem.replace(model_suffix, "")

                test_path = find_test_file(
                    asset_class,
                    dataset_name,
                )

                # ----------------------------------------
                # Load truth + forecast
                # ----------------------------------------

                test_df = pd.read_csv(test_path)
                pred_df = pd.read_csv(point_path)

                y_true = test_df[
                    Y_COLS
                ].to_numpy(dtype=float)

                y_pred = pred_df[
                    Y_COLS
                ].to_numpy(dtype=float)

                # ----------------------------------------
                # Sanity checks
                # ----------------------------------------

                if y_true.shape != y_pred.shape:
                    raise ValueError(
                        f"Shape mismatch: "
                        f"true={y_true.shape}, "
                        f"pred={y_pred.shape}"
                    )

                # ----------------------------------------
                # RMSE
                # ----------------------------------------

                squared_error = (
                    y_true - y_pred
                ) ** 2

                rmse_by_horizon = np.sqrt(
                    np.mean(
                        squared_error,
                        axis=0,
                    )
                )

                overall_rmse = float(
                    np.mean(rmse_by_horizon)
                )

                # ----------------------------------------
                # Save summary
                # ----------------------------------------

                rmse_summary_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name.removeprefix("test_"),
                    "rmse": overall_rmse,
                })

                for h in range(HORIZON):

                    rmse_horizon_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name.removeprefix("test_"),
                        "horizon": h + 1,
                        "rmse": float(
                            rmse_by_horizon[h]
                        ),
                    })

                print(
                    f"    {dataset_name}: "
                    f"RMSE={overall_rmse:.6f}"
                )

            except Exception as exc:

                print(
                    f"    ERROR {point_path.name}: "
                    f"{exc}"
                )


# ==========================================
# SAVE
# ==========================================

rmse_summary_df = pd.DataFrame(
    rmse_summary_rows
)

rmse_horizon_df = pd.DataFrame(
    rmse_horizon_rows
)

rmse_summary_df.to_csv(
    BASE_DIR / "rmse_summary_all_models.csv",
    index=False,
)

rmse_horizon_df.to_csv(
    BASE_DIR / "rmse_by_horizon_all_models.csv",
    index=False,
)

print()
print("Saved RMSE outputs.")

# ARIMA MASE

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

# Where the 80/20 split train/test datasets are
SPLIT_DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

# Where ARIMA saved point forecast CSVs
ARIMA_OUTPUT_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/arima_standardized")

# Where ARIMA saved JSON result files
ARIMA_RESULTS_DIR = (
    ARIMA_OUTPUT_DIR / "results"
)

SUMMARY_PATH = (
    ARIMA_OUTPUT_DIR /
    "arima_mase_summary_all_datasets.csv"
)


# ============================================================
# ASSET CLASS MAPPING
# ============================================================

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ============================================================
# SAME MASE DEFINITION AS ALL OTHER MODELS
# ============================================================

def calculate_mase_from_point_forecast(
    X,
    Y_true,
    Y_pred
):

    naive_forecast = (
        X[:, -1][:, None]
    )

    model_mae_by_horizon = np.mean(
        np.abs(
            Y_true - Y_pred
        ),
        axis=0
    )

    naive_mae_by_horizon = np.mean(
        np.abs(
            Y_true - naive_forecast
        ),
        axis=0
    )

    mase_by_horizon = np.full(
        Y_true.shape[1],
        np.nan,
        dtype=float
    )

    valid_horizons = (
        naive_mae_by_horizon > 0
    )

    mase_by_horizon[
        valid_horizons
    ] = (
        model_mae_by_horizon[
            valid_horizons
        ]
        /
        naive_mae_by_horizon[
            valid_horizons
        ]
    )

    mean_mase = np.nanmean(
        mase_by_horizon
    )

    return (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon
    )

summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    test_dir = (
        SPLIT_DATA_DIR / folder_name
    )

    forecast_dir = (
        ARIMA_OUTPUT_DIR / folder_name
    )

    results_dir = (
        ARIMA_RESULTS_DIR / folder_name
    )

    print("\n" + "=" * 80)
    print(asset_class)
    print("=" * 80)

    # --------------------------------------------------------
    # Loop through ARIMA JSON files
    # --------------------------------------------------------

    json_files = sorted(
        results_dir.glob(
            "*_arima.json"
        )
    )

    print(
        f"Found {len(json_files)} ARIMA results"
    )

    for json_path in json_files:

        name = json_path.stem

        # example:
        #
        # test_USD_AED Historical Data_no_preprocess_
        # lookback-250_horizon-20_slide-1_arima

        prefix = "test_"

        suffix = (
            "_no_preprocess_"
            "lookback-250_"
            "horizon-20_"
            "slide-1_"
            "arima"
        )

        dataset_name = (
            name
            .removeprefix(prefix)
            .removesuffix(suffix)
        )

        # Tether stays excluded
        if "tether" in dataset_name.lower():
            print(
                f"Skipping Tether: {dataset_name}"
            )
            continue

        print(
            f"\nProcessing: {dataset_name}"
        )

        # ====================================================
        # LOAD JSON METADATA
        # ====================================================

        with open(
            json_path,
            "r"
        ) as f:

            metadata = json.load(f)

        # ====================================================
        # MATCH ARIMA POINT FORECAST CSV
        # ====================================================

        forecast_path = (
            forecast_dir /
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1_"
            f"arima.csv"
        )

        if not forecast_path.exists():

            print(
                "Missing forecast:",
                forecast_path
            )

            continue

        # ====================================================
        # MATCH TRUE TEST DATA
        # ====================================================

        test_path = (
            test_dir /
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1.csv"
        )

        if not test_path.exists():

            print(
                "Missing test data:",
                test_path
            )

            continue

        # ====================================================
        # LOAD TEST + FORECAST
        # ====================================================

        test_df = pd.read_csv(
            test_path
        )

        forecast_df = pd.read_csv(
            forecast_path
        )

        x_cols = [
            f"X_{i}"
            for i in range(250)
        ]

        y_cols = [
            f"Y_{i}"
            for i in range(20)
        ]

        X_test = test_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_true = test_df[
            y_cols
        ].to_numpy(dtype=float)

        Y_pred = forecast_df[
            y_cols
        ].to_numpy(dtype=float)

        # ====================================================
        # CHECK ALIGNMENT
        # ====================================================

        if not (
            len(X_test)
            == len(Y_true)
            == len(Y_pred)
        ):

            raise ValueError(
                f"{dataset_name}: "
                f"length mismatch: "
                f"X={len(X_test)}, "
                f"Y={len(Y_true)}, "
                f"pred={len(Y_pred)}"
            )

        # ====================================================
        # CALCULATE MASE
        # ====================================================

        (
            mean_mase,
            mase_by_horizon,
            model_mae_by_horizon,
            naive_mae_by_horizon
        ) = (
            calculate_mase_from_point_forecast(
                X_test,
                Y_true,
                Y_pred
            )
        )

        print(
            f"MASE = {mean_mase:.8f}"
        )

        # ====================================================
        # SAVE MASE BY HORIZON
        # ====================================================

        mase_path = (
            forecast_dir /
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1_"
            f"arima_mase_by_horizon.csv"
        )

        pd.DataFrame({
            "horizon": np.arange(
                1,
                21
            ),
            "model_mae":
                model_mae_by_horizon,
            "naive_mae":
                naive_mae_by_horizon,
            "mase":
                mase_by_horizon
        }).to_csv(
            mase_path,
            index=False
        )

        # ====================================================
        # SUMMARY ROW
        # ====================================================

        order = metadata.get(
            "order",
            [None, None, None]
        )

        summary_rows.append({

            "asset_class":
                asset_class,

            "dataset":
                dataset_name,

            "model":
                "ARIMA",

            "arima_order":
                f"ARIMA({order[0]},{order[1]},{order[2]})",

            "train_aic":
                metadata.get(
                    "train_aic",
                    np.nan
                ),

            "mean_mase":
                mean_mase,

            "n_test_windows":
                len(X_test),

            "training_time_seconds":
                metadata.get(
                    "training_time_seconds",
                    np.nan
                ),

            "testing_time_seconds":
                metadata.get(
                    "testing_time_seconds",
                    np.nan
                ),

            "overall_mae":
                metadata.get(
                    "overall_mae",
                    np.nan
                ),

            "overall_rmse":
                metadata.get(
                    "overall_rmse",
                    np.nan
                ),
        })


# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

arima_summary = pd.DataFrame(
    summary_rows
)

arima_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Saved summary to:\n"
    f"{SUMMARY_PATH}"
)

display(
    arima_summary
)

# LSTM MASE

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOK_BACK = 250
HORIZON = 20
SLIDE = 1

DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/lstm_standardized_train_test")

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_NAME = "LSTM"
MODEL_SUFFIX = "_lstm"


# ============================================================
# MASE
# ============================================================

def calculate_mase_from_point_forecast(
    X,
    Y_true,
    Y_pred
):
    """
    X:      (N, 250) original-scale context
    Y_true: (N, 20)  original-scale targets
    Y_pred: (N, 20)  inverse-transformed LSTM forecasts
    """

    if X.shape[0] != Y_true.shape[0]:
        raise ValueError(
            f"X and Y_true have different N: "
            f"{X.shape[0]} vs {Y_true.shape[0]}"
        )

    if Y_pred.shape != Y_true.shape:
        raise ValueError(
            f"Prediction shape {Y_pred.shape} "
            f"does not match true Y shape {Y_true.shape}"
        )

    # --------------------------------------------
    # Persistence / naive forecast
    # --------------------------------------------

    naive_forecast = X[:, -1][:, None]

    # --------------------------------------------
    # MAE by horizon
    # --------------------------------------------

    model_mae_by_horizon = np.mean(
        np.abs(Y_true - Y_pred),
        axis=0
    )

    naive_mae_by_horizon = np.mean(
        np.abs(Y_true - naive_forecast),
        axis=0
    )

    # --------------------------------------------
    # MASE by horizon
    # --------------------------------------------

    mase_by_horizon = np.full(
        HORIZON,
        np.nan,
        dtype=float
    )

    valid_horizons = (
        naive_mae_by_horizon > 0
    )

    mase_by_horizon[valid_horizons] = (
        model_mae_by_horizon[valid_horizons]
        /
        naive_mae_by_horizon[valid_horizons]
    )

    # --------------------------------------------
    # Dataset-level MASE
    # --------------------------------------------

    mean_mase = np.nanmean(
        mase_by_horizon
    )

    return (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon
    )


# ============================================================
# EVALUATE ALL DATASETS
# ============================================================

summary_rows = []


for folder_name, asset_class in ASSET_FOLDERS.items():

    data_folder = (
        DATA_DIR / folder_name
    )

    results_folder = (
        RESULTS_DIR / folder_name
    )

    print("\n" + "=" * 80)
    print(
        f"ASSET CLASS: {asset_class}"
    )
    print("=" * 80)

    pattern = (
        f"test_*_no_preprocess"
        f"_lookback-{LOOK_BACK}"
        f"_horizon-{HORIZON}"
        f"_slide-{SLIDE}.csv"
    )

    test_paths = sorted(
        data_folder.glob(pattern)
    )

    for test_path in test_paths:

        test_stem = test_path.stem

        print(
            f"\nProcessing: {test_stem}"
        )

        # ====================================================
        # ORIGINAL TEST DATA
        # ====================================================

        test_df = pd.read_csv(
            test_path
        )

        x_cols = [
            f"X_{i}"
            for i in range(LOOK_BACK)
        ]

        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]

        X = test_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_true = test_df[
            y_cols
        ].to_numpy(dtype=float)

        N_test = len(test_df)

        # ====================================================
        # LSTM SAVED POINT FORECAST
        # ====================================================

        prediction_path = (
            results_folder
            / f"{test_stem}{MODEL_SUFFIX}.csv"
        )

        if not prediction_path.exists():

            print(
                f"  MISSING: "
                f"{prediction_path.name}"
            )

            continue

        prediction_df = pd.read_csv(
            prediction_path
        )

        Y_pred = prediction_df[
            y_cols
        ].to_numpy(dtype=float)

        print(
            f"  True Y: {Y_true.shape}"
        )

        print(
            f"  Pred Y: {Y_pred.shape}"
        )

        # ====================================================
        # SANITY CHECKS
        # ====================================================

        if len(prediction_df) != N_test:

            print(
                f"  N mismatch: "
                f"test={N_test}, "
                f"pred={len(prediction_df)}"
            )

            continue

        if not np.isfinite(
            Y_pred
        ).all():

            print(
                "  Non-finite predictions!"
            )

            continue

        # ====================================================
        # MASE
        # ====================================================

        (
            mean_mase,
            mase_by_horizon,
            model_mae_by_horizon,
            naive_mae_by_horizon
        ) = calculate_mase_from_point_forecast(
            X,
            Y_true,
            Y_pred
        )

        print(
            f"  Mean MASE: "
            f"{mean_mase:.10f}"
        )

        # ====================================================
        # SAVE HORIZON RESULTS
        # ====================================================

        mase_df = pd.DataFrame({
            "horizon": np.arange(
                1,
                HORIZON + 1
            ),
            "model_mae": (
                model_mae_by_horizon
            ),
            "naive_mae": (
                naive_mae_by_horizon
            ),
            "mase": mase_by_horizon
        })

        mase_path = (
            results_folder
            / (
                f"{test_stem}"
                f"{MODEL_SUFFIX}"
                f"_mase_by_horizon.csv"
            )
        )

        mase_df.to_csv(
            mase_path,
            index=False
        )

        # ====================================================
        # TRAIN WINDOW COUNT
        # ====================================================

        train_name = (
            "train_"
            + test_path.name[
                len("test_"):
            ]
        )

        train_path = (
            data_folder
            / train_name
        )

        if train_path.exists():

            n_train = len(
                pd.read_csv(
                    train_path,
                    usecols=["X_0"]
                )
            )

        else:

            n_train = np.nan

        # ====================================================
        # CLEAN DATASET NAME
        # ====================================================

        dataset_name = test_stem

        prefix = "test_"

        suffix = (
            f"_no_preprocess"
            f"_lookback-{LOOK_BACK}"
            f"_horizon-{HORIZON}"
            f"_slide-{SLIDE}"
        )

        if dataset_name.startswith(
            prefix
        ):
            dataset_name = (
                dataset_name[
                    len(prefix):
                ]
            )

        if dataset_name.endswith(
            suffix
        ):
            dataset_name = (
                dataset_name[
                    :-len(suffix)
                ]
            )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class": asset_class,
            "dataset": dataset_name,
            "model": MODEL_NAME,
            "mean_mase": mean_mase,
            "n_train_windows": n_train,
            "n_test_windows": N_test
        })


# ============================================================
# MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULTS_DIR
    / "lstm_train_test_summary_all_datasets.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Evaluated datasets: "
    f"{len(summary_df)}"
)

print(
    f"Saved summary to:\n"
    f"{summary_path}"
)

print(summary_df)

# CRPS

In [ ]:
def quantile_crps(y, quantile_forecasts, quantile_levels, quantile_axis=-1):
    """
    Approximate CRPS from quantile forecasts using mean pinball loss.

    Parameters
    ----------
    y : array
        Observations.

    quantile_forecasts : array
        Forecast quantiles.

    quantile_levels : array-like
        Quantile levels, e.g. [0.1, ..., 0.9].

    quantile_axis : int
        Axis corresponding to quantiles.

    Returns
    -------
    crps : array
        Quantile-based CRPS for each observation/horizon.
    """

    y = np.asarray(y, dtype=float)
    qf = np.asarray(quantile_forecasts, dtype=float)
    levels = np.asarray(quantile_levels, dtype=float)

    # Move quantile dimension to the end
    qf = np.moveaxis(qf, quantile_axis, -1)

    # Broadcast observation across quantiles
    error = y[..., None] - qf

    pinball = np.maximum(
        levels * error,
        (levels - 1) * error
    )

    return 2 * np.mean(pinball, axis=-1)

QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

QUANTILE_LEVELS

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

## Moirai CRPS

In [ ]:
PROJECT_DIR = REVIEW_ROOT
DATA_ROOT = PROJECT_DIR / "BForTFin_log_return" / "output_daily"
RESULTS_DIR = PROJECT_DIR / "BForTFin_results" / "standardised" / "output_daily" / "moirai_standardized"

RESULTS_DIR.mkdir(exist_ok=True)

print(Path.cwd())

summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_ROOT / folder_name

    results_dir = (
        RESULTS_DIR /
        folder_name
    )

    for csv_path in sorted(
        input_dir.glob("*.csv")
    ):

        stem = csv_path.stem

        if (
            asset_class == "Crypto"
            and "tether" in stem.lower()
        ):
            continue

        quantiles_path = (
            results_dir /
            f"{stem}_moirai-2.0-r-small_"
            "standardized_zero-shot_"
            "quantiles.npz"
        )

        if not quantiles_path.exists():
            print(
                "Missing quantiles:",
                quantiles_path.name
            )
            continue

        # --------------------------
        # Load truth
        # --------------------------
        df = pd.read_csv(csv_path)

        Y_test = df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        # --------------------------
        # Load Moirai quantiles
        # --------------------------
        data = np.load(
            quantiles_path
        )

        raw_quantiles = data[
            "quantiles"
        ]

        quantile_levels = data[
            "quantile_levels"
        ]

        print(
            stem,
            "raw quantiles:",
            raw_quantiles.shape
        )

        # Expected Moirai layout:
        # (N, 9, 20)

        if raw_quantiles.shape != (
            len(Y_test),
            9,
            20
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"Moirai quantile shape "
                f"{raw_quantiles.shape}"
            )

        if not np.allclose(
            quantile_levels,
            QUANTILE_LEVELS
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"quantile levels "
                f"{quantile_levels}"
            )

        # --------------------------
        # Convert to common shape
        #
        # (N, 9, 20)
        # ->
        # (N, 20, 9)
        # --------------------------
        quantiles = raw_quantiles.transpose(
            0,
            2,
            1
        )

        if quantiles.shape != (
            len(Y_test),
            20,
            9
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"evaluation quantile shape "
                f"{quantiles.shape}"
            )

        # --------------------------
        # qCRPS
        # --------------------------
        crps = quantile_crps(
            Y_test,
            quantiles,
            QUANTILE_LEVELS
        )

        crps_by_horizon = np.mean(
            crps,
            axis=0
        )

        mean_qcrps = np.mean(crps_by_horizon)

        print(
            f"{asset_class} | "
            f"{stem} | "
            f"qCRPS={mean_qcrps:.8f}"
        )

        # --------------------------
        # Save common-shape quantiles
        # --------------------------
        np.savez_compressed(
            results_dir /
            f"{stem}_moirai_standardized_"
            "zero-shot_quantiles.npz",

            quantiles=quantiles,
            quantile_levels=QUANTILE_LEVELS
        )

        # --------------------------
        # Save qCRPS
        # --------------------------
        np.savez_compressed(
            results_dir /
            f"{stem}_moirai_standardized_"
            "zero-shot_crps.npz",

            crps=crps
        )

        # --------------------------
        # Horizon summary
        # --------------------------
        pd.DataFrame({
            "horizon":
                np.arange(1, 21),

            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            results_dir /
            f"{stem}_moirai_standardized_"
            "crps_by_horizon.csv",

            index=False
        )

        # --------------------------
        # Summary
        # --------------------------
        summary_rows.append({
            "asset_class":
                asset_class,

            "dataset":
                stem,

            "model":
                "Moirai",

            "mean_qcrps":
                mean_qcrps,

            "n_windows":
                len(Y_test)
        })

## TimesFM CRPS

In [ ]:
DATA_ROOT = Path(f"BForTFin_log_return/output_{REVIEW_FREQUENCY}")

RESULTS_ROOT = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/timesfm_standardized_train_test")

all_timesfm_summaries = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_ROOT / folder_name
    output_dir = RESULTS_ROOT / folder_name

    test_files = sorted(
        input_dir.glob("*.csv")
    )

    for test_path in test_files:

        stem = test_path.stem

        if (
            asset_class == "Crypto"
            and "tether" in stem.lower()
        ):
            continue

        quantiles_path = (
            output_dir /
            f"{stem}_timesfm-2.5-200m_"
            "standardized_zero-shot_quantiles.npz"
        )

        if not quantiles_path.exists():
            print(
                "Missing:",
                quantiles_path
            )
            continue

        test_df = pd.read_csv(
            test_path
        )

        Y_true = test_df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        raw_output = np.load(
            quantiles_path
        )["quantiles"]

        if raw_output.shape != (
            len(test_df),
            20,
            10
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"TimesFM shape "
                f"{raw_output.shape}"
            )

        quantiles = (
            raw_output[:, :, 1:]
        )

        crps = quantile_crps(
            Y_true,
            quantiles,
            QUANTILE_LEVELS
        )

        crps_by_horizon = (
            crps.mean(axis=0)
        )

        mean_qcrps = crps_by_horizon.mean()

        print(
            f"{asset_class} | "
            f"{stem} | "
            f"{mean_qcrps:.8f}"
        )

        np.savez_compressed(
            output_dir /
            f"{stem}_timesfm_standardized_zero-shot_crps.npz",

            crps=crps
        )

        pd.DataFrame({
            "horizon":
                np.arange(1, 21),

            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            output_dir /
            f"{stem}_timesfm_standardized_"
            "crps_by_horizon.csv",
            index=False
        )

        all_timesfm_summaries.append({
            "asset_class":
                asset_class,

            "dataset":
                stem,

            "model":
                "TimesFM",

            "mean_qcrps":
                mean_qcrps,

            "n_windows":
                len(test_df)
        })

timesfm_summary = pd.DataFrame(
    all_timesfm_summaries
)

timesfm_summary.to_csv(
    RESULTS_ROOT /
    "timesfm_standardized_summary_all_assets.csv",
    index=False
)

## Toto CRPS

In [ ]:
DATA_ROOT = Path(f"BForTFin_log_return/output_{REVIEW_FREQUENCY}")

RESULTS_ROOT = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/toto_standardized_train_test")

summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_ROOT / folder_name
    output_dir = RESULTS_ROOT / folder_name

    for test_path in sorted(
        input_dir.glob("*.csv")
    ):

        stem = test_path.stem

        if (
            asset_class == "Crypto"
            and "tether" in stem.lower()
        ):
            continue

        quantile_path = (
            output_dir /
            f"{stem}_toto_standardized_zero-shot_quantiles.npz"
        )

        if not quantile_path.exists():
            print(
                "Missing:",
                quantile_path
            )
            continue

        df = pd.read_csv(
            test_path
        )

        Y_true = df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        raw = np.load(
            quantile_path
        )["quantiles"]

        # Expected raw Toto layout:
        # (Q, N, 1, H)

        if raw.ndim != 4:
            raise ValueError(
                f"{stem}: unexpected Toto "
                f"shape {raw.shape}"
            )

        if raw.shape[0] != 9:
            raise ValueError(
                f"{stem}: expected 9 "
                f"quantiles, got "
                f"{raw.shape[0]}"
            )

        quantiles = (
            raw[:, :, 0, :]
            .transpose(1, 2, 0)
        )

        expected_shape = (
            len(df),
            20,
            9
        )

        if quantiles.shape != expected_shape:
            raise ValueError(
                f"{stem}: normalized "
                f"shape {quantiles.shape}, "
                f"expected {expected_shape}"
            )

        crps = quantile_crps(
            Y_true,
            quantiles,
            QUANTILE_LEVELS
        )

        crps_by_horizon = crps.mean(axis=0)
        
        mean_qcrps = crps_by_horizon.mean()

        print(
            f"{asset_class} | "
            f"{stem} | "
            f"{mean_qcrps:.8f}"
        )

        # Save CRPS
        np.savez_compressed(
            output_dir /
            f"{stem}_toto_standardized_zero-shot_crps.npz",
            crps=crps
        )

        # Save horizon qCRPS
        pd.DataFrame({
            "horizon":
                np.arange(1, 21),

            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            output_dir /
            f"{stem}_toto_standardized_"
            "crps_by_horizon.csv",
            index=False
        )

        summary_rows.append({
            "asset_class":
                asset_class,

            "dataset":
                stem,

            "model":
                "Toto",

            "mean_qcrps":
                mean_qcrps,

            "n_windows":
                len(df)
        })

toto_summary = pd.DataFrame(
    summary_rows
)

toto_summary.to_csv(
    RESULTS_ROOT /
    "toto_standardized_summary_all_assets.csv",
    index=False
)

# CRPS for all other TSFMs

In [ ]:
import numpy as np

PROJECT_DIR = REVIEW_ROOT
DATA_ROOT = PROJECT_DIR / "BForTFin_log_return"/ "output_daily"
RESULTS_DIR = PROJECT_DIR / "BForTFin_results"/ "standardised" / "output_daily"/ "sundial_standardized_final"

QUANTILE_LEVELS = np.arange(
    0.1,
    1.0,
    0.1
)

def samples_to_quantile_tensor(
    samples,
    quantile_levels=QUANTILE_LEVELS
):
    samples = np.asarray(
        samples,
        dtype=float
    )

    if samples.ndim != 3:
        raise ValueError(
            "Expected samples with shape "
            f"(N, M, H), got {samples.shape}"
        )

    quantiles = np.quantile(
        samples,
        quantile_levels,
        axis=1
    )

    quantiles = np.moveaxis(
        quantiles,
        0,
        -1
    )

    return quantiles

summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_ROOT / folder_name

    results_dir = (
        RESULTS_DIR /
        folder_name
    )

    for csv_path in sorted(
        input_dir.glob("*.csv")
    ):

        stem = csv_path.stem

        if (
            asset_class == "Crypto"
            and "tether" in stem.lower()
        ):
            continue

        samples_path = (
            results_dir /
            f"{stem}_sundial_standardized_"
            "zero-shot_samples.npz"
        )

        if not samples_path.exists():
            print(
                "Missing samples:",
                samples_path.name
            )
            continue

        # --------------------------
        # Load truth
        # --------------------------
        df = pd.read_csv(csv_path)

        Y_test = df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        # --------------------------
        # Load samples
        # --------------------------
        data = np.load(samples_path)

        samples = data["samples"]

        print(
            stem,
            "samples:",
            samples.shape
        )

        if samples.shape != (
            len(Y_test),
            100,
            20
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"samples shape "
                f"{samples.shape}"
            )

        # --------------------------
        # Quantile flattening
        # --------------------------
        quantiles = (
            samples_to_quantile_tensor(
                samples
            )
        )

        if quantiles.shape != (
            len(Y_test),
            20,
            9
        ):
            raise ValueError(
                f"{stem}: unexpected "
                f"quantile shape "
                f"{quantiles.shape}"
            )

        # --------------------------
        # qCRPS
        # --------------------------
        crps = quantile_crps(
            Y_test,
            quantiles,
            QUANTILE_LEVELS
        )


        crps_by_horizon = np.mean(
            crps,
            axis=0
        )

        mean_qcrps = np.mean(crps_by_horizon)

        print(
            f"{asset_class} | "
            f"{stem} | "
            f"qCRPS={mean_qcrps:.8f}"
        )

        # --------------------------
        # Save quantiles
        # --------------------------
        np.savez_compressed(
            results_dir /
            f"{stem}_sundial_standardized_"
            "zero-shot_quantiles.npz",

            quantiles=quantiles,
            quantile_levels=QUANTILE_LEVELS
        )

        # --------------------------
        # Save qCRPS
        # --------------------------
        np.savez_compressed(
            results_dir /
            f"{stem}_sundial_standardized_"
            "zero-shot_crps.npz",

            crps=crps
        )

        # --------------------------
        # Horizon summary
        # --------------------------
        pd.DataFrame({
            "horizon": np.arange(1, 21),
            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            results_dir /
            f"{stem}_sundial_standardized_"
            "crps_by_horizon.csv",
            index=False
        )

        # --------------------------
        # Summary
        # --------------------------
        summary_rows.append({
            "asset_class":
                asset_class,

            "dataset":
                stem,

            "model":
                "Sundial",

            "mean_qcrps":
                mean_qcrps,

            "n_windows":
                len(Y_test)
        })

sundial_summary = pd.DataFrame(
    summary_rows
)

sundial_summary.to_csv(
    RESULTS_DIR /
    "sundial_standardized_summary_all_assets.csv",
    index=False
)

sundial_summary

# MASE

In [ ]:
DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

SUNDIAL_RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/sundial_standardized_train_test")

import numpy as np
import pandas as pd
from pathlib import Path


def calculate_mase_from_point_forecast(X, Y_true, Y_pred):
    """
    X:      (N, 250) original context
    Y_true: (N, 20)  original targets
    Y_pred: (N, 20)  model's saved point forecasts
    """

    if X.shape[0] != Y_true.shape[0]:
        raise ValueError(
            f"X and Y_true have different N: "
            f"{X.shape[0]} vs {Y_true.shape[0]}"
        )

    if Y_pred.shape != Y_true.shape:
        raise ValueError(
            f"Prediction shape {Y_pred.shape} "
            f"does not match true Y shape {Y_true.shape}"
        )

    # Persistence / naive forecast:
    # repeat last context value across all 20 horizons
    naive_forecast = X[:, -1][:, None]

    # Model MAE at each horizon
    model_mae_by_horizon = np.mean(
        np.abs(Y_true - Y_pred),
        axis=0
    )

    # Naive MAE at each horizon
    naive_mae_by_horizon = np.mean(
        np.abs(Y_true - naive_forecast),
        axis=0
    )

    # Horizon-specific MASE
    mase_by_horizon = np.full(
        Y_true.shape[1],
        np.nan,
        dtype=float
    )

    valid_horizons = naive_mae_by_horizon > 0

    mase_by_horizon[valid_horizons] = (
        model_mae_by_horizon[valid_horizons]
        /
        naive_mae_by_horizon[valid_horizons]
    )

    # Dataset-level MASE
    mean_mase = np.nanmean(
        mase_by_horizon
    )

    return (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon
    )

def evaluate_model_mase(
    model_name,
    model_suffix,
    data_dir,
    results_dir,
    asset_folders
):
    """
    model_name:
        e.g. "Chronos-T5-small"

    model_suffix:
        suffix appearing after dataset stem in saved prediction CSV

        e.g.
        "_chronos-t5-small_standardized_zero-shot"

    data_dir:
        directory containing ORIGINAL datasets

    results_dir:
        directory containing this model's saved prediction CSVs
    """

    summary_rows = []

    for folder_name, asset_class in asset_folders.items():

        input_dir = data_dir / folder_name
        model_dir = results_dir / folder_name

        print("\n" + "=" * 70)
        print(f"MODEL: {model_name}")
        print(f"ASSET CLASS: {asset_class}")
        print("=" * 70)

        for original_path in input_dir.glob("*.csv"):

            dataset_name = original_path.stem

            print(f"\nProcessing: {dataset_name}")

            # --------------------------------------------------
            # ORIGINAL DATA
            # --------------------------------------------------

            original_df = pd.read_csv(
                original_path
            )

            X = original_df[
                [f"X_{i}" for i in range(250)]
            ].to_numpy(dtype=float)

            Y_true = original_df[
                [f"Y_{i}" for i in range(20)]
            ].to_numpy(dtype=float)

            N = len(original_df)

            # --------------------------------------------------
            # MODEL OUTPUT CSV
            # --------------------------------------------------

            prediction_path = (
                model_dir
                / f"{dataset_name}{model_suffix}.csv"
            )

            if not prediction_path.exists():

                print(
                    f"⚠️ Missing prediction file: "
                    f"{prediction_path.name}"
                )

                continue

            prediction_df = pd.read_csv(
                prediction_path
            )

            Y_pred = prediction_df[
                [f"Y_{i}" for i in range(20)]
            ].to_numpy(dtype=float)

            print(f"  true Y: {Y_true.shape}")
            print(f"  pred Y: {Y_pred.shape}")

            # --------------------------------------------------
            # SANITY CHECK
            # --------------------------------------------------

            if len(prediction_df) != N:

                print(
                    f"⚠️ N mismatch: "
                    f"original={N}, "
                    f"predictions={len(prediction_df)}"
                )

                continue

            if not np.all(
                np.isfinite(Y_pred)
            ):

                print(
                    f"⚠️ Non-finite predictions "
                    f"in {dataset_name}"
                )

                continue

            # --------------------------------------------------
            # MASE
            # --------------------------------------------------

            (
                mean_mase,
                mase_by_horizon,
                model_mae_by_horizon,
                naive_mae_by_horizon
            ) = calculate_mase_from_point_forecast(
                X,
                Y_true,
                Y_pred
            )

            print(
                f"  Mean MASE: "
                f"{mean_mase:.8f}"
            )

            # --------------------------------------------------
            # SAVE HORIZON RESULTS
            # --------------------------------------------------

            mase_df = pd.DataFrame({
                "horizon": np.arange(1, 21),
                "model_mae": model_mae_by_horizon,
                "naive_mae": naive_mae_by_horizon,
                "mase": mase_by_horizon
            })

            mase_path = (
                model_dir
                / (
                    f"{dataset_name}"
                    f"{model_suffix}"
                    f"_mase_by_horizon.csv"
                )
            )

            mase_df.to_csv(
                mase_path,
                index=False
            )

            # --------------------------------------------------
            # SUMMARY
            # --------------------------------------------------

            summary_rows.append({
                "asset_class": asset_class,
                "dataset": dataset_name,
                "model": model_name,
                "mean_mase": mean_mase,
                "n_windows": N
            })

    summary_df = pd.DataFrame(
        summary_rows
    )

    return summary_df

sundial_mase_summary = evaluate_model_mase(
    model_name="Sundial",
    model_suffix="_sundial_standardized_zero-shot",
    data_dir=DATA_DIR,
    results_dir=SUNDIAL_RESULTS_DIR,
    asset_folders=ASSET_FOLDERS
)

sundial_mase_summary.to_csv(
    SUNDIAL_RESULTS_DIR
    / "sundial_mase_summary_all_datasets.csv",
    index=False
)